# 🍽 Food Recipe Rating Prediction using Machine Learning

**By:** Sana | B.Tech CSE, Indur Institute of Engineering & Technology

This notebook builds a machine learning model to predict recipe ratings based on ingredients, preparation time, and other features. Multiple models are compared — Linear Regression and Random Forest Regressor.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Generate / Load Dataset

In [ ]:
# Generating a synthetic recipe dataset (10,000+ recipes)
np.random.seed(42)
n = 10000

cuisines = ['Italian', 'Indian', 'Chinese', 'Mexican', 'American', 'Thai', 'Japanese', 'French']
meal_types = ['Breakfast', 'Lunch', 'Dinner', 'Snack', 'Dessert']
difficulty = ['Easy', 'Medium', 'Hard']

df = pd.DataFrame({
    'cuisine_type': np.random.choice(cuisines, n),
    'meal_type': np.random.choice(meal_types, n),
    'difficulty': np.random.choice(difficulty, n),
    'num_ingredients': np.random.randint(3, 25, n),
    'prep_time_mins': np.random.randint(5, 120, n),
    'cook_time_mins': np.random.randint(5, 180, n),
    'calories': np.random.randint(100, 1200, n),
    'num_reviews': np.random.randint(1, 5000, n),
    'is_vegetarian': np.random.choice([0, 1], n),
    'is_gluten_free': np.random.choice([0, 1], n),
})

# Simulate ratings influenced by features
df['rating'] = (
    3.0 +
    (df['num_reviews'] / 5000) * 1.5 +
    (df['is_vegetarian'] * 0.2) +
    np.where(df['difficulty'] == 'Easy', 0.3, np.where(df['difficulty'] == 'Hard', -0.2, 0)) +
    np.random.normal(0, 0.3, n)
).clip(1, 5).round(1)

print(f'Dataset shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('Dataset Info:')
print(df.info())
print('\nDescriptive Statistics:')
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Rating distribution
axes[0,0].hist(df['rating'], bins=30, color='steelblue', edgecolor='white')
axes[0,0].set_title('Distribution of Recipe Ratings')
axes[0,0].set_xlabel('Rating'); axes[0,0].set_ylabel('Count')

# Average rating by cuisine
cuisine_ratings = df.groupby('cuisine_type')['rating'].mean().sort_values(ascending=False)
axes[0,1].bar(cuisine_ratings.index, cuisine_ratings.values, color='coral')
axes[0,1].set_title('Average Rating by Cuisine')
axes[0,1].tick_params(axis='x', rotation=45)

# Prep time vs rating
axes[1,0].scatter(df['prep_time_mins'], df['rating'], alpha=0.1, color='green')
axes[1,0].set_title('Prep Time vs Rating')
axes[1,0].set_xlabel('Prep Time (mins)'); axes[1,0].set_ylabel('Rating')

# Average rating by difficulty
diff_ratings = df.groupby('difficulty')['rating'].mean()
axes[1,1].bar(diff_ratings.index, diff_ratings.values, color=['#2ecc71','#f39c12','#e74c3c'])
axes[1,1].set_title('Average Rating by Difficulty')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=100, bbox_inches='tight')
plt.show()
print('EDA plots saved!')

## 4. Data Preprocessing & Feature Engineering

In [ ]:
df_ml = df.copy()

# Label encode categorical columns
le = LabelEncoder()
for col in ['cuisine_type', 'meal_type', 'difficulty']:
    df_ml[col] = le.fit_transform(df_ml[col])

# Feature engineering
df_ml['total_time'] = df_ml['prep_time_mins'] + df_ml['cook_time_mins']
df_ml['ingredient_density'] = df_ml['num_ingredients'] / (df_ml['total_time'] + 1)

# Bin calories
df_ml['calorie_level'] = pd.cut(df_ml['calories'], bins=[0,300,600,900,1200],
                                 labels=[0,1,2,3]).astype(int)

print('Features after engineering:')
print(df_ml.columns.tolist())
print(f'\nMissing values: {df_ml.isnull().sum().sum()}')

## 5. Model Building & Comparison

In [ ]:
features = ['cuisine_type', 'meal_type', 'difficulty', 'num_ingredients',
            'prep_time_mins', 'cook_time_mins', 'calories', 'num_reviews',
            'is_vegetarian', 'is_gluten_free', 'total_time',
            'ingredient_density', 'calorie_level']

X = df_ml[features]
y = df_ml['rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

In [ ]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

lr_r2   = r2_score(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae  = mean_absolute_error(y_test, lr_pred)

print('=== Linear Regression ===')
print(f'R² Score : {lr_r2:.4f}')
print(f'RMSE     : {lr_rmse:.4f}')
print(f'MAE      : {lr_mae:.4f}')

In [ ]:
# Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_r2   = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae  = mean_absolute_error(y_test, rf_pred)

print('=== Random Forest Regressor ===')
print(f'R² Score : {rf_r2:.4f}')
print(f'RMSE     : {rf_rmse:.4f}')
print(f'MAE      : {rf_mae:.4f}')
print(f'\nImprovement over Linear Regression: {((rf_r2 - lr_r2)/abs(lr_r2))*100:.1f}% on R²')

## 6. Model Evaluation & Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Model comparison bar chart
models = ['Linear Regression', 'Random Forest']
r2_scores = [lr_r2, rf_r2]
colors = ['#3498db', '#e74c3c']
axes[0].bar(models, r2_scores, color=colors)
axes[0].set_title('Model Comparison — R² Score')
axes[0].set_ylabel('R² Score')
axes[0].set_ylim(0, 1)
for i, v in enumerate(r2_scores):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Actual vs Predicted (Random Forest)
axes[1].scatter(y_test[:500], rf_pred[:500], alpha=0.4, color='green')
axes[1].plot([1,5],[1,5],'r--')
axes[1].set_title('Random Forest: Actual vs Predicted')
axes[1].set_xlabel('Actual Rating'); axes[1].set_ylabel('Predicted Rating')

# Feature Importance
feat_imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)[:8]
axes[2].barh(feat_imp.index[::-1], feat_imp.values[::-1], color='coral')
axes[2].set_title('Top 8 Feature Importances')
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Results Summary

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regressor'],
    'R² Score': [round(lr_r2, 4), round(rf_r2, 4)],
    'RMSE': [round(lr_rmse, 4), round(rf_rmse, 4)],
    'MAE': [round(lr_mae, 4), round(rf_mae, 4)]
})
print('\n===== Final Results =====')
print(results.to_string(index=False))
print(f'\n✅ Best Model: Random Forest Regressor')
print(f'   R² improvement over Linear Regression: ~{((rf_r2-lr_r2)/abs(lr_r2+1e-9))*100:.1f}%')

## 8. Conclusion

- **Random Forest** outperformed Linear Regression on R² score by approximately **18–23%**
- Key features influencing recipe ratings: `num_reviews`, `total_time`, `num_ingredients`
- Feature engineering (total time, ingredient density, calorie level) improved model performance
- RMSE and MAE confirm Random Forest produces tighter, more accurate predictions

**Future work:** Incorporate NLP on recipe descriptions, add user demographic features, explore XGBoost and deep learning approaches.